In [1]:
# STEP 2 | CUSTOMER LEVEL AGGREGATION

# Build one row per user_id with the five behaviour metrics

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
# Assigning path
path = r'/Users/elia/Desktop/[02] Data Projects/[2] Working Folder/[3] InstaCart Store'

In [4]:
# Load the master file

In [5]:
df_master = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'instacart_master.pkl')
)

print(f"Master loaded: {df_master.shape}")
print()

# ============================================================
# STEP 2 (PRE-AGGREGATION) — REMOVE CORRUPTED-PRICE PRODUCTS
# ============================================================
# Two product_ids have impossibly high prices in products.csv:
#   - 33664: "2 % Reduced Fat Milk"           @ $99,999.00
#   - 21553: "Lowfat 2% Milkfat Cottage Cheese" @ $14,900.00
# These are clearly data entry errors but cannot be reliably
# corrected (we'd be guessing the original values). We drop all
# transaction rows referencing them — ~5,127 rows, 0.0158% of data.
# This MUST happen before customer-level aggregation, otherwise
# avg_product_price gets inflated for affected users and the
# percentile rank in Step 3 becomes meaningless.

rows_before = df_master.shape[0]

corrupt_pids = [33664, 21553]
df_master = df_master[~df_master['product_id'].isin(corrupt_pids)].copy()

rows_after = df_master.shape[0]
rows_dropped = rows_before - rows_after

# Verify no transactions reference the corrupt product_ids
assert df_master['product_id'].isin(corrupt_pids).sum() == 0
# Verify max price is now sane
assert df_master['prices'].max() < 100, \
    f"Max price still {df_master['prices'].max()} — investigate"

print(f"✅ Corrupted-price products removed")
print(f"   Rows before: {rows_before:,}")
print(f"   Rows after:  {rows_after:,}")
print(f"   Rows dropped: {rows_dropped:,} ({rows_dropped/rows_before*100:.4f}%)")
print(f"   Max prices column value now: ${df_master['prices'].max():.2f}")
print()

# Overwrite the pickle so the cleaned version is on disk for any reload
df_master.to_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'instacart_master.pkl')
)
print(f"✅ instacart_master.pkl overwritten with cleaned version")
print(f"   Final shape: {df_master.shape}")

Master loaded: (32403719, 21)

✅ Corrupted-price products removed
   Rows before: 32,403,719
   Rows after:  32,398,592
   Rows dropped: 5,127 (0.0158%)
   Max prices column value now: $25.00

✅ instacart_master.pkl overwritten with cleaned version
   Final shape: (32398592, 21)
